# VA-AFS Colab Runner

Use this notebook from VS Code after selecting `Colab -> Auto Connect` as the kernel. Run cells from top to bottom. The Colab runtime does not use your local workspace, so this notebook clones the GitHub repository into `/content/src` and reads dataset zip files from Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Dataset source:

```text
https://drive.google.com/drive/folders/1JX_Jf2jayPkdh8ii4jwNCrZSe6KdweKh?usp=sharing
```

Copy the four zip files from that shared folder into your own Google Drive. The recommended location is:

```text
/content/drive/MyDrive/AFS/data/
  all_sqe.zip
  nturgbd_skeletons_s001_to_s017.zip
  nturgbd_skeletons_s018_to_s032.zip
  videos.zip
```

The next cell automatically searches your mounted Drive for a folder containing all four zip files, so the exact folder can be different.

In [ ]:
from pathlib import Path

REQUIRED_ZIPS = {
    'all_sqe.zip',
    'nturgbd_skeletons_s001_to_s017.zip',
    'nturgbd_skeletons_s018_to_s032.zip',
    'videos.zip',
}

drive_root = Path('/content/drive')
candidates = {}
for zip_path in drive_root.rglob('*.zip'):
    if zip_path.name in REQUIRED_ZIPS:
        candidates.setdefault(zip_path.parent, set()).add(zip_path.name)

complete_dirs = [path for path, names in candidates.items() if REQUIRED_ZIPS <= names]
if not complete_dirs:
    print('Found matching zip files:')
    for path, names in sorted(candidates.items(), key=lambda item: str(item[0])):
        print(f'- {path}: {sorted(names)}')
    raise FileNotFoundError(
        'Could not find one Drive folder containing all four required zip files. '
        'Copy them into MyDrive/AFS/data or update DATA_DIR manually.'
    )

DATA_DIR = str(sorted(complete_dirs, key=lambda path: str(path))[0])
print('DATA_DIR =', DATA_DIR)
for name in sorted(REQUIRED_ZIPS):
    print(' -', Path(DATA_DIR) / name)

Clone the latest GitHub repo into the Colab runtime and verify that the Colab setup script exists.

In [ ]:
%cd /content
!rm -rf /content/src
!git clone --branch main https://github.com/Kiim-Miin-Su/VA-AFS.git /content/src
%cd /content/src
!git log --oneline -1
!ls -l setup_colab.py VA-AFS/run_colab_pipeline.py requirements-colab.txt

Install dependencies and unzip data. This can take a while on the first run.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, 'setup_colab.py', '--data_dir', DATA_DIR, '--install', '--verify'],
    check=True,
)

Quick smoke test. Run this first to verify that the runtime, data, and dependencies are connected correctly.

In [ ]:
!python VA-AFS/run_colab_pipeline.py \
  --sample_size 200 \
  --num_epoch 2 \
  --batch_size 8 \
  --test_batch_size 8 \
  --num_worker 1

Presentation run: subset 3000, epoch 80.

In [ ]:
!python VA-AFS/run_colab_pipeline.py \
  --sample_size 3000 \
  --num_epoch 80 \
  --batch_size 64 \
  --test_batch_size 64 \
  --num_worker 2

Result files are written under `VA-AFS/outputs/` and `BlockGCN/data/ntu_subset_3000/` in the Colab runtime.